<a href="https://colab.research.google.com/github/carolcv04/CPLUSPLUS/blob/main/packing_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/VasiliBaranov/packing-generation.git

Cloning into 'packing-generation'...
remote: Enumerating objects: 3964, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 3964 (delta 48), reused 91 (delta 40), pack-reused 3857 (from 1)
Receiving objects: 100% (3964/3964), 5.94 MiB | 8.69 MiB/s, done.
Resolving deltas: 100% (1681/1681), done.


In [2]:
%cd /content/packing-generation/_Release

/content/packing-generation/_Release


In [3]:
%pwd

'/content/packing-generation/_Release'

In [4]:
!make

Streaming output truncated to the last 5000 lines.
                 from ../PackingGeneration/Generation/PackingServices/Source/../Headers/ClosestJammingVelocityProvider.h:15,
                 from ../PackingGeneration/Generation/PackingServices/Source/ClosestJammingVelocityProvider.cpp:5:
../Externals/Eigen/Eigen/src/SparseCore/SparseMatrix.h: In function ‘void Eigen::internal::set_from_triplets(const InputIterator&, const InputIterator&, SparseMatrixType&, int)’:
../Externals/Eigen/Eigen/src/SparseCore/SparseMatrix.h:1024:44: warning: typedef ‘Index’ locally defined but not used []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-local-typedefs-Wunused-local-typedefs]8;;]
 1024 |   typedef typename SparseMatrixType::Index Index;
      |                                            ^~~~~
In file included from ../Externals/Eigen/Eigen/SparseCore:59,
                 from ../Externals/Eigen/Eigen/Sparse:17,
                 from ../PackingGeneration/Generation/P

In [5]:
from IPython.core.magic import register_line_cell_magic

@register_line_cell_magic
def writetemplate(line, cell):
    with open(line, 'w') as f:
        f.write(cell.format(**globals()))

In [6]:
from random import randint
num_particles = 250
seed = randint(0, 1000)
print(seed)

293


In [7]:
import numpy as np
from math import pi
#X = np.repeat(1.0, num_particles)
#np.savetxt("diameters.txt", X)

#min_diameter = 0.5
#max_diameter = 10
#X = np.random.uniform(min_diameter, max_diameter, num_particles)
#np.savetxt("diameters.txt", X)
def generate_particle_sizes(num_particles, distribution='uniform', **kwargs):
  """
    distribution: 'uniform', 'normal', 'range'
    **kwargs: distribution-specific parameters
    """
  if distribution == 'uniform':
    min_d = kwargs.get('min_diameter', 0.5)
    max_d = kwargs.get('max_diameter', 5.0)
    X = np.random.uniform(min_d, max_d, num_particles)
  elif distribution == 'normal':
    mean = kwargs.get('mean', 1.0)
    std = kwargs.get('std', 0.2)
    X = np.random.normal(mean, std, num_particles)
    X = np.clip(X, 0.1, None)
  elif distribution == 'range':
    diameter_count = kwargs.get('diameter_count')
    split = num_particles // diameter_count
    remainder = num_particles % diameter_count
    X = []

    for i in range(diameter_count):
      size = kwargs.get(f'size{i+1}')  # FIX: was f'size{diameter_count}'
      count = split + (1 if i < remainder else 0)
      X.extend(np.repeat(size, count))
    X = np.array(X)
  return X

# X = generate_particle_sizes(num_particles, 'range', diameter_count=2, size1=2.0, size2=1.0, size3=3.0)
# X = generate_particle_sizes(num_particles, 'uniform', min_diameter=0.2, max_diameter=6.5)
X = generate_particle_sizes(num_particles, 'uniform', min_diameter=0.2, max_diameter=6.5)


# half = num_particles // 2
# X = np.concatenate([
#     np.repeat(0.9, half),
#     np.repeat(2.0, num_particles - half)
# ])

np.random.shuffle(X)
print(f"Particle distribution: {np.unique(X, return_counts=True)}")
np.savetxt("diameters.txt", X)

particles_vol = sum(4/3 * pi * (X/2)**3)
box_length = (particles_vol * 2)**(1/3)
# box_length = 5.0
print(box_length)

Particle distribution: (array([0.21524361, 0.26220607, 0.26929042, 0.32638168, 0.38950756,
       0.40844399, 0.42718521, 0.42890649, 0.46301185, 0.46911478,
       0.47545683, 0.47840899, 0.49785164, 0.50686697, 0.53160457,
       0.54822727, 0.57397366, 0.58303143, 0.6054223 , 0.61597998,
       0.63485054, 0.68152679, 0.68575572, 0.6998058 , 0.70485151,
       0.73146541, 0.77955563, 0.82401526, 0.83045744, 0.84706918,
       0.87098001, 0.95385256, 0.99115821, 0.99687629, 1.04761106,
       1.05152087, 1.05476634, 1.0788852 , 1.08741053, 1.10547305,
       1.15654952, 1.16687329, 1.18097915, 1.19422899, 1.2223011 ,
       1.22427406, 1.27056494, 1.29470294, 1.29637412, 1.31599543,
       1.33045454, 1.36669451, 1.40651761, 1.4599726 , 1.46552207,
       1.50336505, 1.56089089, 1.56194349, 1.58777325, 1.59385289,
       1.59946262, 1.64911465, 1.67458264, 1.67533031, 1.73073102,
       1.77226637, 1.77526523, 1.78281504, 1.80303431, 1.8076693 ,
       1.82136708, 1.85777374, 1.95664

In [8]:
%%writetemplate generation.conf
Particles count: {num_particles}
Packing size: {box_length} {box_length} {box_length}
Generation start: 1
Seed: {seed}
Steps to write: 1000
Boundaries mode: 1
Contraction rate: 1e-3 #figure out the max and min, parametric study with the different distributions, algorithm to verify that the script is outputting correct output, testing it with the inputs we have ( sucess vs failure ), manually increase the diamter, etc. check for cases where things go wrong (Move paritlces around, etc.), independently be able to check for success vs failure, its difficult to check manually, chekc for other resources

In [9]:
%%time
!rm packing.xyzd packing_init.xyzd packing_prev.xyzd contraction_energies.txt packing.nfo
!./PackingGeneration.exe -fba

rm: cannot remove 'packing.xyzd': No such file or directory
rm: cannot remove 'packing_init.xyzd': No such file or directory
rm: cannot remove 'packing_prev.xyzd': No such file or directory
rm: cannot remove 'contraction_energies.txt': No such file or directory
rm: cannot remove 'packing.nfo': No such file or directory
LittleEndian
The current working directory is /content/packing-generation/_Release.

N is 250
dimensions are 24.626025 24.626025 24.626025
generation mode: start
seed 293
steps to write intermediate state 1000
boundaries mode is 1


Reading particle diameters from a file 'diameters'...
done.
Contraction rate is 0.001
Total volume of particles is 7467.12
Theoretical porosity is 0.5
Global minimum is 0.126889
Step 0. Inner diameter ratio is 0.310001255162488. Outer diameter ratio is 1.062127239898020. Writing int. packing state...Contraction rate: 1.000000e-03
done.
Checking...
Calc. porosity is 0.446099
Theoretical porosity is 0.5
Checking min particle distance in a naive

In [10]:
%%writetemplate generation.conf
Particles count: {num_particles}
Packing size: {box_length} {box_length} {box_length}
Generation start: 0
Seed: {seed}
Steps to write: 1000
Boundaries mode: 1
Contraction rate: 1e-3

In [11]:
!rm packing.nfo
!./PackingGeneration.exe -ls

LittleEndian
The current working directory is /content/packing-generation/_Release.

N is 250
dimensions are 24.626025 24.626025 24.626025
generation mode: continue
seed 293
steps to write intermediate state 1000
boundaries mode is 1


file '/content/packing-generation/_Release/packing.xyzd' was opened
Contraction rate is 0.001
Total volume of particles is 7467.12
Theoretical porosity is 0.5
Global minimum is 1.03472
Time: 3.84219, reduced pressure: 20.5658, actual temperature: 0.234196
Step 0. Inner diameter ratio is 1.038557351457204. Outer diameter ratio is 1.000000000000000. Writing int. packing state...Contraction rate: 1.000000e-03
done.
Time: 5.53792, reduced pressure: 15.4116, actual temperature: 0.236439
Time: 5.10949, reduced pressure: 16.9215, actual temperature: 0.237051
Time: 4.9662, reduced pressure: 17.2406, actual temperature: 0.236473
Time: 4.93932, reduced pressure: 17.4441, actual temperature: 0.236572
Time: 4.41162, reduced pressure: 19.9326, actual temperature: 0.2

In [12]:
!rm packing.nfo
!./PackingGeneration.exe -lsgd

LittleEndian
The current working directory is /content/packing-generation/_Release.

N is 250
dimensions are 24.626025 24.626025 24.626025
generation mode: continue
seed 293
steps to write intermediate state 1000
boundaries mode is 1


file '/content/packing-generation/_Release/packing.xyzd' was opened
Contraction rate is 0.001
Total volume of particles is 7467.12
Theoretical porosity is 0.5
Global minimum is 1.10096
Time: 1.06682e-10, reduced pressure: 1.00507e+12, actual temperature: 0.248729
Suppress growth
Step 0. Inner diameter ratio is 1.100957992222994. Outer diameter ratio is 1.000000000000000. Writing int. packing state...Contraction rate: 1.000000e-03
done.
Time: 1.08391e-10, reduced pressure: 1.23009e+12, actual temperature: 0.2
Time: 1.10175e-10, reduced pressure: 1.21442e+12, actual temperature: 0.2
Time: 1.10279e-10, reduced pressure: 1.14728e+12, actual temperature: 0.2
Time: 1.20104e-10, reduced pressure: 1.04407e+12, actual temperature: 0.2
Time: 1.28476e-10, reduced p

In [13]:
"""https://github.com/VasiliBaranov/packing-generation/issues/30#issue-1103925864"""
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

packing = np.fromfile('packing.xyzd').reshape(-1, 4)

with open('packing.nfo', "r+") as nfo:
    lines = nfo.readlines()
    Theoretical_Porosity = float(lines[2].split()[2])
    Final_Porosity = float(lines[3].split()[2])
    # print(Theoretical_Porosity, Final_Porosity)

    scaling_factor = ((1 - Final_Porosity) / (1 - Theoretical_Porosity)) ** (1 / 3)


    real_diameters = packing[:, 3] * scaling_factor
    actual_density = sum((4/3) * pi * (np.array(real_diameters) / 2)**3) / box_length**3
    print(actual_density)
    packing[:, 3] = real_diameters
    packing.tofile('packing.xyzd')        # updating the packing: this line will modifies diameters in the packing.xyzd

    # update packing.nfo and set TheoreticalPorosity to FinalPorosity to avoid scaling the packing once again the next time running this script.
    lines[3] = lines[3].replace(str(Final_Porosity), str(Theoretical_Porosity))
    nfo.seek(0)
    nfo.writelines(lines)


def ms(x, y, z, radius, resolution=10):
    """Return the coordinates for plotting a sphere centered at (x,y,z)"""
    u, v = np.mgrid[0:2*np.pi:resolution*2j, 0:np.pi:resolution*1j]
    X = radius * np.cos(u)*np.sin(v) + x
    Y = radius * np.sin(u)*np.sin(v) + y
    Z = radius * np.cos(v) + z
    return (X, Y, Z)

df = pd.DataFrame(packing, columns=["x", "y", "z", "d"])
df["r"] = df["d"] / 2
print(df)

# data = []
# for index, row in df.iterrows():
#     (x_pns_surface, y_pns_surface, z_pns_suraface) = ms(row.x, row.y, row.z, row.r)
#     data.append(go.Surface(x=x_pns_surface, y=y_pns_surface, z=z_pns_suraface, opacity=0.95))

# fig = go.Figure(data=data)
# fig.show()

0.667537553204458
             x          y          z         d         r
0     0.500715   3.593644  11.078350  0.585361  0.292681
1     5.984047   7.054763  24.466595  5.432949  2.716475
2    18.577263   9.593354  14.433725  6.589962  3.294981
3     9.148348  19.430665  19.033201  0.755100  0.377550
4    19.745401  17.088417  10.340911  2.992943  1.496471
..         ...        ...        ...       ...       ...
245   2.689990   1.841957  14.854493  4.531599  2.265800
246   5.701458  23.578477   7.254181  0.548195  0.274098
247  12.295181   7.144699  19.432872  4.012121  2.006061
248  13.840118   2.745700  20.702598  4.801644  2.400822
249   1.398032   5.901688   1.637878  4.684645  2.342323

[250 rows x 5 columns]


In [14]:
from scipy.spatial.distance import pdist, squareform
xyz = df[["x", "y", "z"]].values
dm = pdist(xyz)
min(dm / 2) - df.iloc[0]["r"]

np.float64(0.2695982145091095)

In [15]:
from scipy.spatial.distance import pdist

xyz = df[["x", "y", "z"]].values
radii = df["r"].values

dists = pdist(xyz)

# compute pairwise sums of radii
ri_rj = pdist(radii[:, None], metric=lambda u, v: u + v)

margin = dists.min() - ri_rj.min()
print(margin)
overlap = dists < ri_rj - 1e-10
print(overlap)

0.8616925913032935
[False False False ... False False False]


In [16]:
import plotly.express as px
px.histogram(dm)

In [17]:
min(dm)

np.float64(1.1245576393689016)

In [18]:
from sklearn.neighbors import NearestNeighbors
neigh = NearestNeighbors(n_neighbors=5, radius=box_length)
neigh.fit(xyz)
neigh_dist, neigh_ind = neigh.kneighbors(xyz)

In [19]:
neigh_dist[:,1]

array([3.69394101, 4.46439091, 4.33196381, 2.65105236, 2.85089012,
       2.61376618, 2.43224658, 3.61874477, 3.54039338, 1.34372742,
       2.2494657 , 2.2920065 , 2.14045153, 4.69182198, 3.19866113,
       3.9325504 , 4.47543299, 2.29072625, 3.6475904 , 3.02648772,
       3.77450151, 1.34372742, 3.35698705, 3.84250865, 1.1348924 ,
       2.4606022 , 3.39038136, 2.29925624, 3.23766149, 2.13791357,
       4.32028771, 2.04807808, 3.22260065, 2.30523249, 2.72599467,
       2.17950579, 3.83302298, 2.17756125, 1.46280022, 1.6829245 ,
       3.91167115, 2.9389436 , 3.19635115, 3.86448636, 2.90343781,
       1.85820776, 1.85820776, 1.8439351 , 3.83466107, 2.30523249,
       2.295204  , 2.83580396, 2.97137424, 3.78069539, 2.51520846,
       2.20710323, 4.23831069, 3.20720414, 3.83761911, 2.89583028,
       2.36912735, 2.4606022 , 2.83309737, 5.50509841, 4.2857806 ,
       3.23146248, 2.95942058, 3.31537502, 4.12104488, 1.70520111,
       4.73886092, 3.64118836, 2.84362075, 2.56128137, 3.14335

In [20]:
# Check particle bounds vs box boundaries
half_box = box_length / 2

print(f"Box range: [{-half_box:.2f}, {half_box:.2f}]")
print(f"\nParticle positions:")
print(f"X range: [{packing[:, 0].min():.2f}, {packing[:, 0].max():.2f}]")
print(f"Y range: [{packing[:, 1].min():.2f}, {packing[:, 1].max():.2f}]")
print(f"Z range: [{packing[:, 2].min():.2f}, {packing[:, 2].max():.2f}]")

# Check how much particles exceed boundaries
print(f"\nMax particle radius: {(packing[:, 3]/2).max():.3f}")
print(f"\nParticles exceeding bounds:")
exceeds = ((packing[:, 0].max() + packing[:, 3].max()/2) > half_box) or \
          ((packing[:, 1].max() + packing[:, 3].max()/2) > half_box) or \
          ((packing[:, 2].max() + packing[:, 3].max()/2) > half_box)
print(f"Exceeds: {exceeds}")

Box range: [-12.31, 12.31]

Particle positions:
X range: [0.03, 24.44]
Y range: [0.02, 24.61]
Z range: [0.04, 24.51]

Max particle radius: 3.573

Particles exceeding bounds:
Exceeds: True


In [23]:
import plotly.graph_objects as go
import numpy as np


def create_sphere(center, radius, color, name=None, resolution=10):
    """Create a sphere mesh for Plotly"""
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)

    x = radius * np.outer(np.cos(u), np.sin(v)) + center[0]
    y = radius * np.outer(np.sin(u), np.sin(v)) + center[1]
    z = radius * np.outer(np.ones(np.size(u)), np.cos(v)) + center[2]

    return go.Surface(
        x=x, y=y, z=z,
        colorscale=[[0, color], [1, color]],
        showscale=False,
        name=name,
        hoverinfo='skip',
        opacity=0.8
    )

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=packing[:, 0],
    y=packing[:, 1],
    z=packing[:, 2],
    mode='markers',
    marker=dict(
        size=packing[:, 3] * 3,   # scale diameter
        color=packing[:, 3],      # color by size
        colorscale='Turbo',
        opacity=0.8,
        showscale=True,
        line=dict(width=0)
    ),
    hoverinfo='skip'
))

fig.update_layout(
    scene=dict(aspectmode='data'),
    title="Fast Particle Packing View"
)

fig.show()
# Add each particle as an actual 3D sphere
colors_map = ['rgb(31, 119, 180)', 'rgb(255, 127, 14)', 'rgb(44, 160, 44)',
              'rgb(214, 39, 40)', 'rgb(148, 103, 189)', 'rgb(140, 86, 75)',
              'rgb(227, 119, 194)', 'rgb(127, 127, 127)']

for i in range(len(packing)):
    center = packing[i, :3]
    radius = packing[i, 3] / 2
    color = colors_map[i % len(colors_map)]

    fig.add_trace(create_sphere(center, radius, color, name=f"Particle {i}"))

# Draw box edges
max_radius = (packing[:, 3] / 2).max()
box_min = 0 - max_radius
box_max = box_length + max_radius

corners = np.array([
    [box_min, box_min, box_min],
    [box_max, box_min, box_min],
    [box_max, box_max, box_min],
    [box_min, box_max, box_min],
    [box_min, box_min, box_max],
    [box_max, box_min, box_max],
    [box_max, box_max, box_max],
    [box_min, box_max, box_max],
])

edges = [
    [0, 1], [1, 2], [2, 3], [3, 0],
    [4, 5], [5, 6], [6, 7], [7, 4],
    [0, 4], [1, 5], [2, 6], [3, 7]
]

for edge in edges:
    points = corners[edge]
    fig.add_trace(go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='lines',
        line=dict(color='red', width=3),
        showlegend=False,
        hoverinfo='skip'
    ))

fig.update_layout(
    width=1000,
    height=800,
    scene=dict(
        xaxis=dict(range=[box_min, box_max]),
        yaxis=dict(range=[box_min, box_max]),
        zaxis=dict(range=[box_min, box_max]),
        aspectmode='data',
    ),
    title=f'Particle Packing ({len(packing)} particles)',
    showlegend=False
)
fig.show()

print("---------------------------------------- PARAMETER INFO ----------------------------------------")
print(f"Box size: {box_length:.4f}")
print(f"Number of particles: {len(packing)}")
print(f"Particle diameter distribution: {np.unique(packing[:, 3], return_counts=True)}")

# DIAMETER STATISTICS
print(f"\nDiameter Statistics:")
print(f"  Min diameter: {packing[:, 3].min():.4f}")
print(f"  Max diameter: {packing[:, 3].max():.4f}")
print(f"  Mean diameter: {packing[:, 3].mean():.4f}")
# print(f"  Std dev: {packing[:, 3].std():.4f}")
# print(f"  Coefficient of variation: {packing[:, 3].std() / packing[:, 3].mean():.4f}")

# POROSITY METRICS
print(f"\nPorosity Metrics:")
print(f"  Theoretical porosity: {Theoretical_Porosity:.6f}")
print(f"  Final porosity: {Final_Porosity:.6f}")
print(f"  Porosity error: {abs(Final_Porosity - Theoretical_Porosity):.6f} ({abs(Final_Porosity - Theoretical_Porosity)/Theoretical_Porosity*100:.2f}%)")

# PACKING EFFICIENCY
packing_fraction = 1 - Final_Porosity
print(f"\nPacking Efficiency:")
print(f"  Packing fraction: {packing_fraction:.6f}")
print(f"  Actual density: {actual_density:.6f}")

# DENSITY & VOLUME
particle_volumes = (4/3) * np.pi * (packing[:, 3] / 2)**3
total_particle_volume = particle_volumes.sum()
print(f"\nVolume Analysis:")
print(f"  Total particle volume: {total_particle_volume:.4f}")
print(f"  Box volume: {box_length**3:.4f}")
print(f"  Void volume: {box_length**3 - total_particle_volume:.4f}")

# SPATIAL DISTRIBUTION
print(f"\nSpatial Distribution:")
print(f"  Min distance between particles: {min(dm):.4f}")
print(f"  Max distance between particles: {max(dm):.4f}")
print(f"  Mean distance between particles: {np.mean(dm):.4f}")
print(f"  Std dev distance: {np.std(dm):.4f}")

# COORDINATION NUMBER (average neighbors)
from scipy.spatial.distance import pdist, squareform
distances = squareform(pdist(packing[:, :3]))
contact_distance = packing[:, 3].max() + 0.01  # slightly more than max diameter
coordination_number = (distances > 0) & (distances < contact_distance)
avg_neighbors = coordination_number.sum(axis=1).mean()
print(f"\nNeighborhood Analysis:")
print(f"  Average coordination number: {avg_neighbors:.2f}")

# BOUNDARY CHECKS
print(f"\nBoundary Checks:")
particles_touching_boundary = 0
for i in range(len(packing)):
    x, y, z, d = packing[i]
    r = d / 2
    if x - r < 0 or x + r > box_length or \
       y - r < 0 or y + r > box_length or \
       z - r < 0 or z + r > box_length:
        particles_touching_boundary += 1
print(f"  Particles touching boundary: {particles_touching_boundary}/{len(packing)}")

# OVERLAP DETECTION
print(f"\nOverlap Detection:")
overlapping_pairs = 0
for i in range(len(packing)):
    for j in range(i+1, len(packing)):
        dist = np.linalg.norm(packing[i, :3] - packing[j, :3])
        min_dist = (packing[i, 3] + packing[j, 3]) / 2
        if dist < min_dist - 0.001:  # small tolerance
            overlapping_pairs += 1
print(f"  Overlapping particle pairs: {overlapping_pairs}")
print(f"  Status: {'PASS' if overlapping_pairs == 0 else 'FAIL - OVERLAPS DETECTED'}")

# SUMMARY/STATUS
print(f"\n" + "="*100)
print(f"SIMULATION STATUS: {'SUCCESS' if overlapping_pairs == 0 and Final_Porosity <= Theoretical_Porosity * 1.1 else 'FAILED'}")
print(f"="*100)

---------------------------------------- PARAMETER INFO ----------------------------------------
Box size: 24.6260
Number of particles: 250
Particle diameter distribution: (array([0.23700936, 0.28872073, 0.29652147, 0.35938588, 0.42889513,
       0.44974645, 0.4703828 , 0.47227815, 0.50983229, 0.51655236,
       0.52353573, 0.52678641, 0.54819513, 0.5581221 , 0.58536121,
       0.60366482, 0.63201472, 0.64198843, 0.66664349, 0.67826878,
       0.69904756, 0.75044379, 0.75510035, 0.77057119, 0.77612713,
       0.80543227, 0.85838544, 0.90734089, 0.91443452, 0.93272606,
       0.9590548 , 1.05030753, 1.09138559, 1.09768189, 1.15354703,
       1.15785221, 1.16142587, 1.18798367, 1.19737108, 1.21726011,
       1.27350151, 1.28486924, 1.3004015 , 1.31499118, 1.34590199,
       1.34807446, 1.39904634, 1.42562521, 1.42746538, 1.44907083,
       1.46499206, 1.50489667, 1.54874675, 1.60760719, 1.61371782,
       1.65538754, 1.71873048, 1.71988952, 1.74833123, 1.75502565,
       1.76120264, 1.81

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import Normalize

class EnhancedFluidSim:
    def __init__(self, particles, domain_size, num_fluid_particles=2000):
        """
        Enhanced fluid simulation with pressure, diffusion, and injection

        particles: your packing array (x, y, z, diameter)
        domain_size: box_length from your packing
        num_fluid_particles: initial number of fluid particles
        """
        self.particles = particles
        self.domain_size = domain_size
        self.time_step = 0

        # Fluid particles (represent liquid)
        self.create_fluid_particles(num_fluid_particles)

        # Simulation parameters
        self.dt = 0.0001      # time step
        self.g = 9.81         # gravity
        self.nu = 0.001       # viscosity
        self.rho = 1000       # fluid density
        self.h = 0.5          # smoothing length
        self.pressure_strength = 50
        self.diffusion_dist = 1.0

    def create_fluid_particles(self, n_fluid):
        """Create random fluid particles in domain"""
        self.fluid_pos = np.random.uniform(0, self.domain_size, (n_fluid, 3))
        self.fluid_vel = np.zeros((n_fluid, 3))
        self.fluid_pressure = np.zeros(n_fluid)

        # Concentration: 1 = original fluid, 0 = new solution
        self.concentration = np.ones(n_fluid)

    def check_collision_with_solids(self, fluid_pos):
        """Check if fluid particles overlap with solid particles"""
        solid_pos = self.particles[:, :3]
        solid_radii = self.particles[:, 3] / 2

        valid = np.ones(len(fluid_pos), dtype=bool)

        for i, fpos in enumerate(fluid_pos):
            for j, spos in enumerate(solid_pos):
                dist = np.linalg.norm(fpos - spos)
                if dist < solid_radii[j] + 0.1:  # 0.1 margin
                    valid[i] = False
                    break

        return valid

    def apply_pressure_forces(self):
        """Fluid particles repel each other (pressure/incompressibility)"""
        n = len(self.fluid_pos)

        for i in range(n):
            force = np.array([0.0, 0.0, 0.0])

            for j in range(n):
                if i == j:
                    continue

                diff = self.fluid_pos[i] - self.fluid_pos[j]
                dist = np.linalg.norm(diff)

                if dist < self.h and dist > 0.01:
                    # Repulsive force (pressure)
                    direction = diff / dist
                    force_mag = (self.h - dist) / self.h * self.pressure_strength
                    force += direction * force_mag

            self.fluid_vel[i] += force * self.dt

    def apply_viscosity(self):
        """Apply viscous damping"""
        self.fluid_vel *= (1 - self.nu * self.dt)

    def simulate_diffusion(self):
        """Solution diffuses through fluid via mixing"""
        n = len(self.fluid_pos)

        for i in range(n):
            for j in range(i+1, n):
                dist = np.linalg.norm(self.fluid_pos[i] - self.fluid_pos[j])

                if dist < self.diffusion_dist:
                    # Mix concentrations
                    weight = 1.0 - (dist / self.diffusion_dist)
                    avg = (self.concentration[i] + self.concentration[j]) / 2

                    self.concentration[i] = avg * weight * 0.1 + self.concentration[i] * (1 - weight * 0.1)
                    self.concentration[j] = avg * weight * 0.1 + self.concentration[j] * (1 - weight * 0.1)

    def inject_solution(self, num_new=50, concentration_value=0.0):
        """Add new solution particles at top of domain"""
        new_pos = np.random.uniform(
            [0, 0, self.domain_size - 2],
            [self.domain_size, self.domain_size, self.domain_size],
            (num_new, 3)
        )
        new_vel = np.random.uniform(-0.5, 0.5, (num_new, 3))
        new_conc = np.ones(num_new) * concentration_value

        self.fluid_pos = np.vstack([self.fluid_pos, new_pos])
        self.fluid_vel = np.vstack([self.fluid_vel, new_vel])
        self.concentration = np.hstack([self.concentration, new_conc])

    def simulate_step(self):
        """One simulation step"""
        # Apply forces
        self.apply_pressure_forces()

        # Gravity
        self.fluid_vel[:, 2] -= self.g * self.dt

        # Viscosity damping
        self.apply_viscosity()

        # Update positions
        self.fluid_pos += self.fluid_vel * self.dt

        # Boundary conditions
        for dim in range(3):
            # Lower boundary
            out_lower = self.fluid_pos[:, dim] < 0
            self.fluid_vel[out_lower, dim] *= -0.5
            self.fluid_pos[out_lower, dim] = 0

            # Upper boundary
            out_upper = self.fluid_pos[:, dim] > self.domain_size
            self.fluid_vel[out_upper, dim] *= -0.5
            self.fluid_pos[out_upper, dim] = self.domain_size

        # # Remove fluid inside solids
        # valid = self.check_collision_with_solids(self.fluid_pos)
        # self.fluid_pos = self.fluid_pos[valid]
        # self.fluid_vel = self.fluid_vel[valid]
        # self.concentration = self.concentration[valid]

        # Diffusion
        self.simulate_diffusion()

        self.time_step += 1

    def get_velocity_magnitude(self):
        """Get velocity magnitude for each particle"""
        return np.linalg.norm(self.fluid_vel, axis=1)

    def visualize_step(self, step, show_velocity=False, show_concentration=True):
        """Visualize current state"""
        fig = plt.figure(figsize=(14, 10))
        ax = fig.add_subplot(111, projection='3d')

        # Plot solid particles
        solid_scatter = ax.scatter(
            self.particles[:, 0],
            self.particles[:, 1],
            self.particles[:, 2],
            s=self.particles[:, 3] * 20,
            c='darkred',
            alpha=0.8,
            label='Solid particles',
            edgecolors='black',
            linewidth=0.3
        )

        # Plot fluid particles
        if len(self.fluid_pos) > 0:
            if show_concentration and show_velocity:
                # Color by velocity magnitude
                velocities = self.get_velocity_magnitude()
                norm = Normalize(vmin=0, vmax=np.percentile(velocities, 95))

                fluid_scatter = ax.scatter(
                    self.fluid_pos[:, 0],
                    self.fluid_pos[:, 1],
                    self.fluid_pos[:, 2],
                    s=8,
                    c=velocities,
                    cmap='hot',
                    alpha=0.6,
                    norm=norm,
                    label='Fluid (color=velocity)'
                )
                cbar = plt.colorbar(fluid_scatter, ax=ax, label='Velocity magnitude')

            elif show_concentration:
                # Color by concentration (mixing)
                norm = Normalize(vmin=0, vmax=1)

                fluid_scatter = ax.scatter(
                    self.fluid_pos[:, 0],
                    self.fluid_pos[:, 1],
                    self.fluid_pos[:, 2],
                    s=8,
                    c=self.concentration,
                    cmap='RdYlBu_r',
                    alpha=0.6,
                    norm=norm,
                    label='Fluid (Red=old, Blue=new)'
                )
                cbar = plt.colorbar(fluid_scatter, ax=ax, label='Concentration (1=old, 0=new)')
            else:
                fluid_scatter = ax.scatter(
                    self.fluid_pos[:, 0],
                    self.fluid_pos[:, 1],
                    self.fluid_pos[:, 2],
                    s=8,
                    c='blue',
                    alpha=0.5,
                    label='Fluid particles'
                )

        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_xlim(0, self.domain_size)
        ax.set_ylim(0, self.domain_size)
        ax.set_zlim(0, self.domain_size)
        ax.legend(loc='upper right')

        title = f'Enhanced Fluid Simulation - Step {step}\n'
        title += f'Fluid particles: {len(self.fluid_pos)} | '

        if len(self.fluid_pos) > 0:
            avg_conc = np.mean(self.concentration)
            title += f'Avg concentration: {avg_conc:.3f}'

        ax.set_title(title)

        plt.tight_layout()
        plt.show()


# Main execution
def run_simulation(packing, box_length, num_steps=200, injection_interval=20):
    """
    Run complete simulation with injection and visualization

    Parameters:
    -----------
    packing : array
        Particle packing data (N, 4) with columns [x, y, z, diameter]
    box_length : float
        Size of simulation domain
    num_steps : int
        Number of simulation steps to run
    injection_interval : int
        Steps between solution injections
    """

    print("=" * 60)
    print("ENHANCED FLUID SIMULATION")
    print("=" * 60)
    print(f"Domain size: {box_length:.2f}³")
    print(f"Solid particles: {len(packing)}")
    print(f"Initial fluid particles: 2000")
    print(f"Running for {num_steps} steps")
    print(f"Injecting new solution every {injection_interval} steps")
    print("=" * 60)

    # Initialize simulation
    sim = EnhancedFluidSim(packing, box_length, num_fluid_particles=2000)

    print(f"\nInitial state:")
    print(f"  Fluid particles: {len(sim.fluid_pos)}")
    print(f"  Avg concentration: {np.mean(sim.concentration):.3f}")

    # Run simulation
    for step in range(num_steps):
        sim.simulate_step()

        # Inject solution periodically
        if step % injection_interval == 0 and step > 0:
            sim.inject_solution(num_new=100, concentration_value=0.0)
            print(f"\nStep {step}: Injected new solution")
            print(f"  Total fluid particles: {len(sim.fluid_pos)}")
            print(f"  Avg concentration: {np.mean(sim.concentration):.3f}")

        # Visualize
        if step % 50 == 0 or step == num_steps - 1:
            print(f"\nStep {step}: Visualizing...")

            # Show concentration evolution
            sim.visualize_step(
                step,
                show_velocity=False,
                show_concentration=True
            )

            if step > 0:
                # Show velocity field
                sim.visualize_step(
                    step,
                    show_velocity=True,
                    show_concentration=False
                )

    print("\n" + "=" * 60)
    print("SIMULATION COMPLETE")
    print(f"Final fluid particles: {len(sim.fluid_pos)}")
    print(f"Final avg concentration: {np.mean(sim.concentration):.3f}")
    print("=" * 60)

    return sim


# Run if in Jupyter/Colab
if __name__ == "__main__":
    print("Load your packing data first, then run:")
    print("sim = run_simulation(packing, box_length, num_steps=200, injection_interval=20)")

In [ ]:
# sim = run_simulation(x
#     box_length,
#     num_steps=200,           # Total steps to run
#     injection_interval=100    # Inject new solution every 20 steps
# )